### Workspace for Matt

## Data load, clean and match

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Locate data directory (relative to notebook: 'data/', with fallback to '../Data_Raw/')
data_dir = Path('data') if Path('data').exists() else Path('../Data_Raw')
if not data_dir.exists():
    data_dir = Path('Data_Raw')

# Load raw CSV datasets into DataFrames (using first-row headers as column names)
Harris1 = pd.read_csv(data_dir / 'HarrisPartI.csv', na_values=['NA', 'XXXX', 'NONE'])
Harris3 = pd.read_csv(data_dir / 'HarrisPartIII.csv', na_values=['NA', 'XXXX', 'NONE'])
vandenBerg = pd.read_csv(data_dir / 'vandenBerg_table2.csv', na_values=['NA', 'XXXX', 'NONE'])
Krause = pd.read_csv(data_dir / 'Krause21.csv', na_values=['NA', 'XXXX', 'NONE'])

# Strip leading '#' from '#NGC' so vandenBerg.NGC works as a direct attribute
vandenBerg.columns = [col.lstrip('#') for col in vandenBerg.columns]

# Lowercase aliases for convenience
harris1, harris3, vandenberg, krause = Harris1, Harris3, vandenBerg, Krause

# Verify access format: Harris1.ID[4]
print(f"Harris1.ID[4] = '{Harris1.ID[3]}'")
print(f"Loaded shapes: Harris1 {Harris1.shape}, Harris3 {Harris3.shape}, vandenBerg {vandenBerg.shape}, Krause {Krause.shape}")


Harris1.ID[4] = 'Whiting 1'
Loaded shapes: Harris1 (157, 11), Harris3 (157, 13), vandenBerg (55, 13), Krause (61, 8)


In [2]:
"""Run after the notebook cell that loads Harris1, Harris3, vandenBerg and Krause.

Creates master (two-level columns) and match_report, without changing raw frames.
"""
import re
from collections import defaultdict

# Abreviation dictionary
NAME_MAP = {
    'RUPRECHT106': 'RUP106',
    'PALOMAR12': 'PAL12',
    'TER8': 'TERZAN8',
}

# normalise name if not na, conform to upper case, check for null identifier. Check dictionary to see for abreviation. returns correct case condition, null identifier or abbreviation
def normalise_name(value):
    if pd.isna(value):
        return None
    key = re.sub(r'\s+', '', str(value)).upper()
    if key in {'', 'NA', 'NAN', 'NONE', 'XXXX'}:
        return None
    return NAME_MAP.get(key, key)

# matches ngc ID
def ngc_number(value, bare_number=False):
    key = normalise_name(value)
    if key is None:
        return pd.NA
    pattern = r'(\d+)(?:\.0+)?' if bare_number else r'NGC(\d+)'
    found = re.fullmatch(pattern, key)
    return int(found.group(1)) if found else pd.NA

# Work on copies; protect original csv files.
sources = {
    'h1': Harris1.copy(),
    'h3': Harris3.copy(),
    'k': Krause.copy(),
    'vb': vandenBerg.copy(),
}
sources['vb'] = sources['vb'].rename(columns={'#NGC': 'NGC'})
primary_column = {'h1': 'ID', 'h3': 'ID', 'k': 'Object', 'vb': 'NGC'}
alias_column = {'h1': 'Name', 'h3': None, 'k': 'AltName', 'vb': 'Name'}

for source, frame in sources.items():
    if not frame.columns.is_unique:
        raise ValueError(f'{source}: duplicate column names need review.')
    original = frame[primary_column[source]]
    numbers = original.map(lambda x: ngc_number(x, bare_number=(source == 'vb')))
    # In vb this converts its existing NGC field to nullable integers.
    # Other sources gain a derived NGC fieldretain original IDs.
    frame['NGC'] = pd.array(numbers, dtype='Int64')

# Build a lookup using Harris primary IDs AND its listed alternative names.
harris_lookup = defaultdict(set)
display_names = {}
for _, row in sources['h1'].iterrows():
    if normalise_name(row['ID']) is None:
        raise ValueError('Harris I contains a missing primary ID.')
    cluster_id = str(row['ID']).strip()
    if cluster_id in display_names:
        raise ValueError(f'Duplicate Harris I ID: {cluster_id}')
    display_names[cluster_id] = (
        str(row['Name']).strip() if normalise_name(row['Name']) else cluster_id
    )
    for value in [row['ID'], row['Name']]:
        key = normalise_name(value)
        if key is not None:
            harris_lookup[key].add(cluster_id)

def lookup_unique(key):
    candidates = harris_lookup.get(key, set())
    if len(candidates) > 1:
        raise ValueError(f'Ambiguous Harris name {key!r}: {sorted(candidates)}')
    return next(iter(candidates)) if candidates else None

records = []
indexed_sources = {}
for source, frame in sources.items():
    common_ids = []
    for row_position, (_, row) in enumerate(frame.iterrows()):
        number = row['NGC']
        primary = row[primary_column[source]]
        alias = row[alias_column[source]] if alias_column[source] else pd.NA
        primary_key = (
            f'NGC{number}' if pd.notna(number) else
            normalise_name(alias if source == 'vb' else primary)
        )
        alias_key = normalise_name(alias)
        primary_match = lookup_unique(primary_key)
        alias_match = lookup_unique(alias_key)

        if primary_match and alias_match and primary_match != alias_match:
            raise ValueError(f'{source} row {row_position}: ID and alias disagree.')
        if pd.notna(number) and alias_match:
            if normalise_name(alias_match) != f'NGC{number}':
                raise ValueError(f'{source} row {row_position}: NGC number conflicts with alias.')

        cluster_id = primary_match or alias_match
        if cluster_id:
            method = 'NGC' if pd.notna(number) and primary_match else (
                'primary name' if primary_match else 'Harris alias'
            )
        else:
            # Source-qualified keys to avoid accidentally merging unresolved objects.
            label = primary_key or alias_key or f'ROW{row_position}'
            cluster_id = f'UNMATCHED:{source}:{label}'
            method = 'unmatched - review'
            display_names[cluster_id] = (
                str(alias).strip() if normalise_name(alias) else
                str(primary).strip() if normalise_name(primary) else cluster_id
            )

        common_ids.append(cluster_id)
        records.append({
            'source': source, 'source_row': row_position,
            'original_id': primary, 'original_alias': alias,
            'cluster_id': cluster_id, 'NGC': number, 'match_method': method,
        })

    if pd.Index(common_ids).has_duplicates:
        repeated = pd.Series(common_ids)[pd.Series(common_ids).duplicated(keep=False)]
        raise ValueError(f'{source}: multiple rows map to one object; review {repeated.tolist()}')
    frame.index = pd.Index(common_ids, name='cluster_id')
    indexed_sources[source] = frame

# Outer alignment keeps every source row
master = pd.concat(indexed_sources, axis=1, join='outer', sort=False)
master.columns.names = ['source', 'quantity']

match_report = pd.DataFrame(records)
match_report['NGC'] = pd.array(match_report['NGC'], dtype='Int64')
meta = pd.DataFrame(index=master.index)
meta['cluster_id'] = master.index.astype('string')
meta['preferred_name'] = pd.Series(display_names, dtype='string').reindex(master.index)
meta['NGC'] = pd.Series(pd.NA, index=master.index, dtype='Int64')
for frame in indexed_sources.values():
    meta['NGC'] = meta['NGC'].fillna(frame['NGC'].reindex(master.index))
meta['matched_to_harris'] = master.index.isin(sources['h1']['ID'].str.strip())
meta.columns = pd.MultiIndex.from_product([['meta'], meta.columns], names=['source', 'quantity'])
master = pd.concat([meta, master], axis=1)

print(f'Master table: {len(master)} clusters, {master.shape[1]} columns')
print('\nMatching summary:')
print(pd.crosstab(match_report['source'], match_report['match_method']).to_string())
unmatched = match_report[match_report['match_method'] == 'unmatched - review']
print('\nUnmatched rows (retained in master):')
print(unmatched[['source', 'original_id', 'original_alias', 'cluster_id']].to_string(index=False))

# Notebook access examples:
# master['h1']                         # All Harris I columns
# master['h3']['v_r']                   # Radial velocities
# master.loc['NGC 104', ('vb', 'Age')]   # One cluster and measurement
# master['meta']['NGC']                 # Nullable integer NGC numbers
# master[[('vb', 'Age'), ('vb', 'FeH')]].dropna()  # Plot-specific sample


Master table: 158 clusters, 52 columns

Matching summary:
match_method  NGC  primary name  unmatched - review
source                                             
h1            104            53                   0
h3            104            53                   0
k              58             3                   0
vb             52             2                   1

Unmatched rows (retained in master):
source original_id original_alias         cluster_id
    vb        <NA>          Arp21 UNMATCHED:vb:ARP21


In [3]:
# save master csv
master.to_csv(data_dir / "master_catalogue.csv", index=True)
""" example on how to load csv
master = pd.read_csv(
    data_dir / "master_catalogue.csv",
    header=[0, 1],
    index_col=0
)

# Restore nullable integer types for NGC numbers
for source in ["meta", "h1", "h3", "k", "vb"]:
    master[(source, "NGC")] = master[(source, "NGC")].astype("Int64")

"""

' example on how to load csv\nmaster = pd.read_csv(\n    data_dir / "master_catalogue.csv",\n    header=[0, 1],\n    index_col=0\n)\n\n# Restore nullable integer types for NGC numbers\nfor source in ["meta", "h1", "h3", "k", "vb"]:\n    master[(source, "NGC")] = master[(source, "NGC")].astype("Int64")\n\n'

### Data cleaning

## Data Visualisation

### Exploratory

### Age Metalicity


## Candidate Classification

## Extension

## Video Stuff